# Solver speed benchmark: UMFPACK, Intel PARDISO and NVIDIA cuDSS on the solver's own matrices

**What it does, in one session of about 45-60 minutes:**
1. Builds the P2′ release solver exactly as a production run does, after checking the code against the release's content digest.
2. Runs the stiff clot `fsf_G40_klo_v250` for about 100 steps with a small capture library loaded at run time. The library writes out the matrices the solver hands to UMFPACK. **The solver itself is not changed.**
3. Times UMFPACK (called the way the solver calls it), Intel PARDISO and, on a GPU runtime, NVIDIA cuDSS on those matrices, at 1, 2, 4 and all cores.

**Before Run all,** choose the runtime for what this session measures (Runtime → Change runtime type):
- **The CPU runtime you would use for production** (High-RAM if your plan has it). This prices PARDISO on that machine; cuDSS is skipped.
- **A100 GPU**, only to measure cuDSS (done once already, session 20261007-074244).

The `GITHUB_TOKEN` secret is the same one the main notebook uses.

**Results** go to Drive → `OpenFOAM_Blood/bench/<stamp>/`:
- `results.md` and `results.json`;
- the logs;
- the captured matrices, about 70 MB.

When the determinism cell prints `DETERMINISM_COMPLETE <stamp>`, send the stamp, or download that folder into `bench/`.

**It does not touch production.** It uses its own branch and its own folders, and writes nothing to `OpenFOAM_Blood/runs`. If this runtime's image differs from the usual CPU runtime's, the setup cell builds OpenFOAM once (about an hour extra) and caches it on Drive, as the main notebook would.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
# SET-UP AND CAPTURE. Leave the pins alone: they are written by build_notebook.py.
from datetime import datetime
from pathlib import Path
import hashlib, importlib.util, json, os, re, shlex, shutil, signal, subprocess, sys, time

REPO_URL = 'https://github.com/alirezakianim/openfoam-blood.git'
BENCH_BRANCH = 'solver-bench'
# The P2' release (release-2d3eed1e): the solver and case must hash to this.
RELEASE_REPO_DIGEST = '8497d46a7010a30e85b9458690ed169121cd4225848fe4a0c50594de45dc42c2'
KIT_SHA256 = {
    "umfdump.c": "c83c410fd23f55a7840239f6f7c5acf505a9b115f084f5d9856e7d1d46344b6d",
    "bench_common.h": "37cde687284de22f96ea122d2054e4bf4f60a896a2d792bec291e99125d193fb",
    "bench_umf.cpp": "a1fd1b0fb45e17f98617dd1cb54fea02b1dc580588ec9d5000aab4356fa5186e",
    "bench_pardiso.cpp": "3ac296d94b780516a5175d700579f41859863254eae227abe32cba2f770868bb",
    "bench_cudss.cpp": "e892fa49f6538e1a4cc0aabb57d8e599d99e85dafdfc5c01aab08f8555f71032",
    "run_bench.py": "9c1ddf0e2c921a9ed097ae69002c404fee8a41419534cd29e7ca332c17a13256",
    "plugin_selftest.sh": "f8bf8ce10e8d4b100e2a1b6c80c4f6656694472745cdd60483b8d46f993fa73d"
}
CAPTURE_CHILD = 'fsf_G40_klo_v250'   # the stiff clot whose Colab timing is the baseline
CAPTURE_CALLS = '1,2,100'            # UMFPACK factorisations to capture (about one per step)
CAPTURE_TIMEOUT_MIN = 45

STAMP = datetime.now().strftime('%Y%m%d-%H%M%S')
CONTENT = Path('/content')
WORK = CONTENT / 'solver_bench'
DRIVE_OUT = CONTENT / 'drive' / 'MyDrive' / 'OpenFOAM_Blood' / 'bench' / STAMP
shutil.rmtree(WORK, ignore_errors=True)
WORK.mkdir(parents=True)
q = shlex.quote

def bash(command, log_name=None):
    """Runs one bash command; keeps its output in WORK/<log_name>."""
    r = subprocess.run(['bash', '-c', command], capture_output=True, text=True)
    if log_name:
        (WORK / log_name).write_text(r.stdout + r.stderr)
    return r

def refuse(message, log_name=None):
    if log_name and (WORK / log_name).is_file():
        tail = (WORK / log_name).read_text().splitlines()[-40:]
        message += '\n--- last lines of ' + log_name + ' ---\n' + '\n'.join(tail)
    raise SystemExit('BENCH REFUSED: ' + message)

# 0. The hardware this session got.
hw = bash('echo "nproc $(nproc)"; lscpu | grep -E "^Model name|^CPU\\(s\\)|^Thread|^Core|^Socket"; '
          'free -g | head -2; nvidia-smi --query-gpu=name,memory.total,driver_version '
          '--format=csv,noheader 2>/dev/null || echo "GPU: none"; '
          'echo "--- core/thread layout"; lscpu -e=CPU,CORE,SOCKET,ONLINE', 'hardware.txt')
print(hw.stdout)

# 1. The code: the bench branch of the private repo. The token reaches git only
#    through the environment and an askpass helper, exactly as in the main notebook.
try:
    from google.colab import userdata
    token = userdata.get('GITHUB_TOKEN')
    why = 'the secret is empty'
except Exception as error:   # SecretNotFoundError, NotebookAccessError
    token, why = None, type(error).__name__
if not token:
    refuse(f'no GITHUB_TOKEN ({why}). Use the same Colab secret as the main notebook '
           '(key icon -> Secrets, with Notebook access on).')
clone = WORK / 'openfoam-blood'
askpass = WORK / 'git_askpass.sh'
askpass.write_text('#!/bin/sh\ncase "$1" in\n  Username*) echo x-access-token ;;\n'
                   '  *) printf \'%s\\n\' "$GH_TOKEN" ;;\nesac\n')
askpass.chmod(0o700)
cloned = subprocess.run(
    ['git', '-c', 'credential.helper=', 'clone', '--quiet', '--depth', '1',
     '--branch', BENCH_BRANCH, REPO_URL, str(clone)],
    env=dict(os.environ, GH_TOKEN=token, GIT_ASKPASS=str(askpass), GIT_TERMINAL_PROMPT='0'),
    capture_output=True, text=True)
if cloned.returncode != 0:
    message = (cloned.stdout + cloned.stderr).replace(token, '<token>')
    del token
    refuse(f'git clone of branch {BENCH_BRANCH} failed (return code {cloned.returncode}).\n{message}')
del token
commit = subprocess.run(['git', '-C', str(clone), 'rev-parse', 'HEAD'],
                        capture_output=True, text=True).stdout.strip()

# 2. Verify before spending the session: the solver and case are the P2' release
#    (the release's own digest code, taken from its notebook), and the kit is the
#    tested one (sha256 per file).
main_nb = json.loads((clone / 'notebooks' / 'AM2020_main_case_colab.ipynb').read_text())
cell = next(''.join(c['source']) for c in main_nb['cells']
            if c['cell_type'] == 'code' and '# >>> repo_digest' in ''.join(c['source']))
block = re.search(r'^# >>> repo_digest\n.*?^# <<< repo_digest\n', cell, re.S | re.M)
if block is None:
    refuse('the release notebook carries no repo_digest block')
digest_ns = {}
exec(block.group(0), digest_ns)
digest = digest_ns['repo_digest'](clone, (clone / 'tools' / 'manifest.txt').read_text())
if digest != RELEASE_REPO_DIGEST:
    refuse(f'the solver/case content is not the P2 release: digest {digest}, expected {RELEASE_REPO_DIGEST}')
kit = clone / 'tools' / 'solver_bench'
for name, want in KIT_SHA256.items():
    path = kit / name
    got = hashlib.sha256(path.read_bytes()).hexdigest() if path.is_file() else 'missing'
    if got != want:
        refuse(f'kit file {name} is not the tested one: {got} != {want}')
print(f'code: branch {BENCH_BRANCH} commit {commit[:12]} | solver+case digest {digest[:12]} = P2 release | kit verified')

# 3. OpenFOAM through the production bootstrap: the same apt packages (UMFPACK and
#    OpenBLAS) and the same Drive cache. ensure_openfoam() only checks that its
#    package path exists, so it is pointed at a file of the verified clone.
spec = importlib.util.spec_from_file_location(
    'bench_bootstrap',
    clone / 'cases' / 'AM2020_paper_cylinder_platelet_initial_condition_gate' / 'scripts'
    / 'run_platelet_initial_condition_colab.py')
bootstrap = importlib.util.module_from_spec(spec)
spec.loader.exec_module(bootstrap)
bootstrap.PACKAGE_DRIVE = clone / 'tools' / 'manifest.txt'
t0 = time.time()
bootstrap.ensure_openfoam()
OF = f'source {q(str(bootstrap.OF_BASHRC))}'
print(f'OpenFOAM ready in {time.time() - t0:.0f} s')

# 4. The solver, built the way the runner builds it.
solver = clone / 'user-applications' / 'phaseClotAxisymmetricFluid'
t0 = time.time()
built = bash(f'{OF} && cd {q(str(solver))} && bash ./Allwmake', 'log.Allwmake')
if built.returncode != 0:
    refuse('the solver build failed', 'log.Allwmake')
print(f'solver built in {time.time() - t0:.0f} s')

# 5. The capture works only if the solver reaches UMFPACK through the shared library.
linked = bash(f'{OF} && ldd "$FOAM_USER_LIBBIN/libphaseClotAxisymmetricFluid.so"', 'ldd.txt')
umf = [line.strip() for line in linked.stdout.splitlines() if 'libumfpack' in line]
if linked.returncode != 0 or not umf or 'not found' in umf[0]:
    refuse('the solver does not load libumfpack dynamically, so the capture library would be bypassed', 'ldd.txt')
print('solver links', umf[0])

# 6. The capture library: its own self-test (a host linked to UMFPACK), then a
#    plugin loaded RTLD_LAZY|RTLD_GLOBAL, the way OpenFOAM loads the solver.
shim_run = subprocess.run([sys.executable, str(kit / 'run_bench.py'), '--shim-only',
                           '--out', str(WORK / 'shim')], capture_output=True, text=True)
(WORK / 'log.shim_selftest').write_text(shim_run.stdout + shim_run.stderr)
shim = WORK / 'shim' / 'bin' / 'libumfdump.so'
if shim_run.returncode != 0 or 'SELFTEST ALL PASS' not in shim_run.stdout + shim_run.stderr \
        or not shim.is_file():
    refuse('the capture library failed its self-test', 'log.shim_selftest')
plugin = bash(f'bash {q(str(kit / "plugin_selftest.sh"))} {q(str(shim))} {q(str(WORK / "plugin_selftest"))}',
              'log.plugin_selftest')
if plugin.returncode != 0 or 'PLUGIN_SELFTEST PASS' not in plugin.stdout:
    refuse('the capture library failed the plugin-path self-test', 'log.plugin_selftest')
print('capture library: self-test PASS, plugin path PASS')

# 7. The capture: the case staged as the runner stages it, then the child's own
#    Allrun with the library preloaded. The run is stopped once every listed
#    factorisation is on disk.
cases = WORK / 'cases'
shutil.copytree(clone / 'cases', cases,
                ignore=shutil.ignore_patterns('__pycache__', '*.pyc', 'runs', 'postProcessing', 'log.*'))
gate = cases / 'AM2020_planar_mural_detachment'
shutil.copy2(WORK / 'log.Allwmake', gate / 'log.Allwmake')
capture = WORK / 'capture'
env = dict(os.environ, AM2020_CONCURRENCY='1', LD_PRELOAD=str(shim),
           UMFDUMP_DIR=str(capture), UMFDUMP_CALLS=CAPTURE_CALLS)
with open(WORK / 'log.capture_Allrun', 'w') as allrun_log:
    proc = subprocess.Popen(['bash', '-c', f'{OF} && cd {q(str(gate))} && bash ./Allrun {q(CAPTURE_CHILD)}'],
                            env=env, stdout=allrun_log, stderr=subprocess.STDOUT, start_new_session=True)
    solver_log = gate / 'runs' / CAPTURE_CHILD / 'log.phaseClotAxisymmetricFluid'
    t0, last_print, status = time.time(), 0.0, None
    while status is None:
        time.sleep(10)
        if (capture / 'DONE').is_file():
            status = 'DONE'
        elif (capture / 'FAILED').is_file():
            status = 'FAILED'
        elif proc.poll() is not None:
            status = f'Allrun exited ({proc.returncode}) before the capture finished'
        elif time.time() - t0 > CAPTURE_TIMEOUT_MIN * 60:
            status = f'TIMEOUT after {CAPTURE_TIMEOUT_MIN} min'
        if time.time() - last_print > 120:
            last_print = time.time()
            steps = 0
            if solver_log.is_file():
                steps = sum(1 for line in solver_log.read_text(errors='replace').splitlines()
                            if line.startswith('Time = '))
            done = sorted(p.name for p in capture.glob('call*.x')) if capture.is_dir() else []
            print(f'  {time.time() - t0:5.0f} s: solver steps {steps}, captured {done}', flush=True)
    if proc.poll() is None:
        os.killpg(proc.pid, signal.SIGTERM)
        try:
            proc.wait(timeout=30)
        except subprocess.TimeoutExpired:
            os.killpg(proc.pid, signal.SIGKILL)
            proc.wait()
print('capture:', status)
if solver_log.is_file():
    shutil.copy2(solver_log, WORK / 'log.capture_solver')
if status != 'DONE':
    if (capture / 'FAILED').is_file():
        print((capture / 'FAILED').read_text())
    refuse(f'the capture did not complete ({status})', 'log.capture_Allrun')

# 8. Keep the capture and the logs on Drive before benchmarking.
DRIVE_OUT.mkdir(parents=True)
shutil.copytree(capture, DRIVE_OUT / 'capture')
for name in ['hardware.txt', 'log.Allwmake', 'ldd.txt', 'log.shim_selftest', 'log.plugin_selftest',
             'log.capture_Allrun', 'log.capture_solver']:
    if (WORK / name).is_file():
        shutil.copy2(WORK / name, DRIVE_OUT / name)
(DRIVE_OUT / 'bench_source.json').write_text(json.dumps(
    {'stamp': STAMP, 'branch': BENCH_BRANCH, 'commit': commit, 'release_repo_digest': digest,
     'kit_sha256': KIT_SHA256, 'capture_child': CAPTURE_CHILD, 'capture_calls': CAPTURE_CALLS}, indent=2) + '\n')
(WORK / 'CAPTURE_OK').write_text(STAMP + '\n')
print(f'CAPTURE_OK {STAMP}: {sorted(p.name for p in capture.glob("call*.json"))} saved to {DRIVE_OUT}')


In [ ]:
# THE BENCHMARK: the three solvers on the captured matrices.
if not (WORK / 'CAPTURE_OK').is_file():
    raise SystemExit('BENCH REFUSED: the set-up cell did not finish its capture in this session')
gpu = subprocess.run(['bash', '-c', 'nvidia-smi -L'], capture_output=True, text=True).returncode == 0
packages = ['mkl==2026.1.0', 'mkl-include==2026.1.0']
r = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q'] + packages, capture_output=True, text=True)
(WORK / 'log.pip_mkl').write_text(r.stdout + r.stderr)
print('MKL (PARDISO):', 'installed' if r.returncode == 0 else 'install FAILED, PARDISO will be skipped')
if gpu:
    r = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--no-deps', 'nvidia-cudss-cu12==0.8.0.10'],
                       capture_output=True, text=True)
    (WORK / 'log.pip_cudss').write_text(r.stdout + r.stderr)
    print('cuDSS:', 'installed' if r.returncode == 0 else 'install FAILED, cuDSS will be skipped')
else:
    print('cuDSS: no GPU on this runtime, skipped')

captures = sorted((p.with_suffix('') for p in (WORK / 'capture').glob('call*.json')),
                  key=lambda p: int(p.name[4:]))
command = [sys.executable, str(kit / 'run_bench.py'), '--out', str(WORK / 'results'),
           '--reps', '3', '--threads', '1,2,4,nproc']
for prefix in captures:
    command += ['--capture', str(prefix)]
if gpu:
    command += ['--cudss-mt']
t0 = time.time()
with subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True) as run:
    for line in run.stdout:
        print(line, end='', flush=True)
print(f'benchmark exit {run.returncode} after {time.time() - t0:.0f} s')
results = WORK / 'results'
if results.is_dir():
    shutil.copytree(results, DRIVE_OUT / 'results', dirs_exist_ok=True)
for name in ['log.pip_mkl', 'log.pip_cudss']:
    if (WORK / name).is_file():
        shutil.copy2(WORK / name, DRIVE_OUT / name)
table = results / 'results.md'
print(table.read_text() if table.is_file() else 'no results.md was written: see the output above')
print(f'BENCH_COMPLETE {STAMP} -> {DRIVE_OUT} (the determinism cell runs next)')


In [ ]:
# DETERMINISM: does PARDISO give bit-identical answers when it factors the same matrix
# again? Default mode against CNR mode (iparm33 = threads, MKL_CBWR=AUTO), at all and at
# half the threads, on the last capture.
sys.path.insert(0, str(kit))
import run_bench
mkl, why = run_bench.find_mkl()
exe = WORK / 'results' / 'bin' / 'bench_pardiso'
if mkl is None or not exe.is_file():
    raise SystemExit(f'BENCH REFUSED: no PARDISO build to test ({why or exe})')
last = max((p.with_suffix('') for p in (WORK / 'capture').glob('call*.json')), key=lambda p: int(p.name[4:]))
nproc = os.cpu_count() or 1
thread_counts = sorted({nproc, max(1, nproc // 2), min(2, nproc)}, reverse=True)
rows = []
for t in thread_counts:
    for mode in ('default', 'CNR'):
        env = dict(os.environ, MKL_NUM_THREADS=str(t), OMP_NUM_THREADS=str(t),
                   LD_LIBRARY_PATH=mkl['libdir'] + ':' + os.environ.get('LD_LIBRARY_PATH', ''))
        env.pop('MKL_CBWR', None)
        extra = []
        if mode == 'CNR':
            env['MKL_CBWR'] = 'AUTO'
            extra = [f'iparm33={t}']
        r = subprocess.run([str(exe), str(last), str(t), '4', '4'] + extra, env=env, capture_output=True, text=True)
        tag = f'determinism_{last.name}_{mode}_t{t}'
        (WORK / 'results' / 'logs' / f'{tag}.log').write_text(r.stdout + r.stderr)
        res = next((json.loads(line[len('RESULT_JSON '):]) for line in r.stdout.splitlines()
                    if line.startswith('RESULT_JSON ')), {'error': f'no result (exit {r.returncode})'})
        res.update(mode=mode, threads=t, rc=r.returncode)
        rows.append(res)
        print(f"{mode:7} thr {t:2}: refactor {res.get('refactor_s', {}).get('med', float('nan')):.3f} s, "
              f"analysis {res.get('analysis_s', {}).get('med', float('nan')):.3f} s | identical: fresh "
              f"{res.get('repro_fresh_identical')}/{res.get('repro_fresh_compared')}, refactor "
              f"{res.get('repro_refac_identical')}/{res.get('repro_refac_compared')}, maxdiff "
              f"{res.get('repro_maxdiff', float('nan')):.2e} | relres2 {res.get('relres2', float('nan')):.2e} "
              f"| status {res.get('status')} rc {r.returncode}", flush=True)
(WORK / 'results' / 'determinism.json').write_text(json.dumps(rows, indent=1) + '\n')
shutil.copytree(WORK / 'results', DRIVE_OUT / 'results', dirs_exist_ok=True)
shutil.copy2(WORK / 'hardware.txt', DRIVE_OUT / 'hardware.txt')
print(f'DETERMINISM_COMPLETE {STAMP} -> {DRIVE_OUT}')


In [ ]:
# OPTIONAL: download the results and logs (not the 70 MB of matrices) as one zip.
from google.colab import files
bundle = WORK / f'bench-{STAMP}'
shutil.rmtree(bundle, ignore_errors=True)
shutil.copytree(DRIVE_OUT, bundle, ignore=shutil.ignore_patterns('capture'))
files.download(shutil.make_archive(str(bundle), 'zip', root_dir=str(bundle)))
